# Temperature–mortality response and humidity modification

Native analysis for Figure 2. Relative risk is derived from each member's fitted temperature branch with the original standard age distribution and the final-day baseline.

[Original analysis notebook](https://github.com/ClimSocAna/sari-weather-mortality/blob/4402a80d3e490ad7964a3be12597043761bebda1/notebooks/02_temperature_humidity_response.ipynb).

In [ ]:
from pathlib import Path
import sys

repo = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "configs" / "training.example.yaml").is_file())
sys.path.insert(0, str(repo))

import matplotlib.pyplot as plt
import numpy as np
import torch
from notebooks.utils import ensemble
from sari_t.mean_alignment import apply_mean_alignment
from sari_t.models.interfaces import MortalityModelInputs
from sari_t.reference import build_reference_weather

from notebooks.utils import ANALYSIS_PERIOD, DEVICE, checkpoint_paths
from sari_t.inference import InferenceModel
from sari_t.run.checkpoint import load_checkpoint, load_run_config, restore_predictor


## Load models for inspection

Run the next cell to load member `000` for each setup into memory. Inspect `model` for the hybrid predictor or `models[setup]` for another predictor. Set `inspection_member` to a value from 0 through 9 to inspect another member. The analysis cells evaluate all ten members in a separate inference workspace.

In [ ]:
inspection_member = 0
setups = ['default', 'no_disease', 'tas_only']
checkpoints = {setup: checkpoint_paths(setup) for setup in setups}
models = {
    setup: restore_predictor(load_checkpoint(paths[inspection_member]), device="cpu").eval()
    for setup, paths in checkpoints.items()
}
model = models["default"]
model

## Evaluate the fitted response

In [ ]:
temperatures = np.arange(-5, 32.5, 0.5)
std_age = torch.tensor([88569, 86870, 85970, 84670, 82171, 79272, 76073,
                        71475, 65877, 60379, 53681, 45484, 37187, 29590,
                        22092, 15195, 9097, 4398, 1500, 400, 50]).float()
std_age = torch.cat((std_age[:6].sum().reshape(1), std_age[6:-2],
                     std_age[-2:].sum().reshape(1)))[:, None]

def response(humidities):
    @torch.no_grad()
    def summarize(model, result):
        baseline = model.baseline_rate[-1] * std_age.to(model.baseline_rate.device)
        curves = {}
        for humidity in humidities:
            weather = torch.cat([
                build_reference_weather(
                    model.data.model_inputs.temperature, model.weather_variables,
                    temperature, humidity, kernel_days=model.model_config.kernel_T_days,
                )
                for temperature in temperatures
            ])
            factor = model.predictor.effect_model(
                MortalityModelInputs(weather, None, None)
            ).temperature_effect.abs()
            mortality = apply_mean_alignment(factor * baseline, model.alignment_factor)
            curves[humidity] = mortality.sum(dim=(-1, -2)).squeeze(0)
        return curves
    return summarize

summaries = {
    "default": response(range(30, 81, 10)),
    "no_disease": response([50]),
    "tas_only": response([50]),
}

estimates_by_setup = {}
for setup, summarize_member in summaries.items():
    paths = checkpoints[setup]
    inputs = load_run_config(paths[0].parents[1] / "config.yaml").data_config
    inference_model = InferenceModel.from_checkpoint(
        paths[0], data_config=inputs, period=ANALYSIS_PERIOD, device=DEVICE,
    )
    estimates_by_setup[setup] = ensemble(inference_model, paths, summarize_member)
    del inference_model

hybrid = estimates_by_setup["default"]
weather_only = estimates_by_setup["no_disease"]
tas_only = estimates_by_setup["tas_only"]

## Figure 2 · Relative risk curves

In [ ]:
def excess_risk(curve):
    return (curve / curve.min() - 1) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
axes[0].plot(temperatures, excess_risk(hybrid[50]), label="Weather and SARI")
axes[0].plot(temperatures, excess_risk(weather_only[50]), label="Weather only")
axes[1].plot(temperatures, excess_risk(tas_only[50]), label="Temperature only")
for humidity, curve in hybrid.items():
    axes[1].plot(temperatures, excess_risk(curve), label=f"{humidity}% RH")
for ax in axes:
    ax.set(xlabel="Air temperature (°C)", ylabel="Risk above model minimum (%)")
    ax.legend(frameon=False)
fig.tight_layout()